# Model Development 

We'll start a baseline using Logistic Regression that help us classify claim/no claim using sigmoid. 

We'll use its metrics to compare against Tree-based ensembles (RF, XGBoost, LightGBM) to see where multi-feature classification excels for underwriting.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, f1_score,
    roc_auc_score, precision_recall_curve, average_precision_score,
    roc_curve, precision_score, recall_score
)
import warnings
warnings.filterwarnings('ignore')

from imblearn.over_sampling import SMOTE, ADASYN, BorderlineSMOTE
from imblearn.under_sampling import RandomUnderSampler, TomekLinks
from imblearn.combine import SMOTETomek, SMOTEENN
from imblearn.pipeline import Pipeline as ImbPipeline
import kagglehub
import os


In [ ]:
# Housekeeping
path = kagglehub.dataset_download("litvinenko630/insurance-claims")

# List all files in the downloaded directory
files = os.listdir(path)
file_path = os.path.join(path, "Insurance claims data.csv")
df = pd.read_csv(file_path)

from helpers.preprocess import *

df_processed = preprocess_data(df)
df_processed, _ = train_test_split(
    df_processed, 
    train_size=0.90,      # Keep 10% of the data
    stratify=df_processed['claim_status'], 
    random_state=42
)


# Separate features and target
X = df_processed.drop('claim_status', axis=1)
y = df_processed['claim_status']

# Train-test split 
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Standardize features 
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training: {X_train.shape[0]} samples")
print(f"Test: {X_test.shape[0]} samples")
print(f"Features: {X_train.shape[1]}")
print(f"\nClass distribution in training:")
print(y_train.value_counts())


# Baseline helps us identify coefficients on which features has the most influence on labels
